# Travel RAG Retrieval Pipeline — Stage-by-Stage Notebook

This notebook is designed for Jupyter testing of the full retrieval pipeline:

1. User Query
2. Query Rewriting
3. Semantic Query Parsing
4. User Memory Retrieval
5. Metadata-aware Filtering
6. Vector Retrieval
7. BM25 Retrieval
8. Score Fusion + Metadata Boost
9. Top-K Pruning
10. CrossEncoder Reranking
11. Retrieval Confidence
12. Evidence Construction
13. Conditional Tool Planning
14. Structured Itinerary Drafting
15. Citation Extraction
16. Frontend JSON Response

> Keep production logic in `.py` modules later. This notebook is for debugging and evaluation.

## 0. Imports and Models

In [39]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

# If notebook is running from /notebooks, move to project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python path added:", PROJECT_ROOT)

Project root: E:\Thesis_all\be
Python path added: E:\Thesis_all\be


In [40]:
from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np

from pydantic import BaseModel, Field

from sqlalchemy import text
from sqlalchemy.orm import Session

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

from db.session import SessionLocal
from db.full_model import RagChunkORM

In [41]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME
)

## 1. Shared Schemas

In [42]:
class QueryLocation(BaseModel):
    country: Optional[str] = None
    city: Optional[str] = None
    province: Optional[str] = None


class QueryConstraints(BaseModel):
    budget: Optional[str] = None
    duration_days: Optional[int] = None

    date_from: Optional[str] = None
    date_to: Optional[str] = None

    near_place: Optional[str] = None
    max_distance_km: Optional[float] = None


class ParsedQuery(BaseModel):
    intent: Optional[str] = None

    location: QueryLocation = Field(
        default_factory=QueryLocation
    )

    place_types: list[str] = Field(
        default_factory=list
    )

    activities: list[str] = Field(
        default_factory=list
    )

    travel_styles: list[str] = Field(
        default_factory=list
    )

    suitable_for: list[str] = Field(
        default_factory=list
    )

    constraints: QueryConstraints = Field(
        default_factory=QueryConstraints
    )


class UserTravelMemory(BaseModel):
    preferred_travel_styles: list[str] = Field(
        default_factory=list
    )

    preferred_activities: list[str] = Field(
        default_factory=list
    )

    budget_level: Optional[str] = None

    avoid: list[str] = Field(
        default_factory=list
    )

## 2. Test Query

In [43]:
query = (
    "Plan family-friendly activities in Da Nang for 3 days. "
    "I like cultural attractions and local food."
)

conversation_history = [
    {
        "role": "user",
        "content": "I am planning a Vietnam trip."
    }
]

print(query)

Plan family-friendly activities in Da Nang for 3 days. I like cultural attractions and local food.


## 3. Query Rewriting

In [44]:
import json
from typing import Any

from data_building.extract_metadata.extractor import DEEPSEEK_METADATA_MODEL, get_deepseek_client


def rewrite_query(
    query: str,
    conversation_history: list[dict],
    model: str = DEEPSEEK_METADATA_MODEL,
) -> str:
    """
    Rewrite the current user query into a standalone retrieval query
    using recent conversation history.

    If no history exists, return the original query.
    """

    query = query.strip()

    if not query:
        raise ValueError("Query cannot be empty")

    if not conversation_history:
        return query

    deepseek_client = get_deepseek_client()

    # Keep only recent turns to avoid unnecessary tokens.
    recent_history = conversation_history[-6:]

    history_text = "\n".join(
        f"{item.get('role', 'unknown')}: "
        f"{item.get('content', '')}"
        for item in recent_history
    )

    system_msg = """
        You are a query rewriting system for a tourism RAG application.

        Your task is to rewrite the user's latest query into a clear,
        standalone retrieval query using the conversation history.

        RULES:
        - Resolve references such as:
        "there", "that place", "it", "those", "what about kids?"
        - Preserve the user's original intent.
        - Preserve explicit locations, dates, duration, budget, travel style,
        activities, and constraints.
        - Use conversation history only to resolve missing context.
        - Do not add preferences or facts that the user did not express.
        - Do not answer the question.
        - Do not make the query unnecessarily long.
        - If the current query is already standalone, keep it mostly unchanged.

        Return valid JSON only.

        Required JSON format:
        {
        "rewritten_query": "standalone retrieval query"
        }
    """.strip()

    user_prompt = f"""
        Conversation history:
        {history_text}

        Current user query:
        {query}

        Return the rewritten query as JSON.
    """.strip()

    response = deepseek_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_msg,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0,
        max_tokens=300,
        response_format={
            "type": "json_object",
        },
    )

    raw_text = response.choices[0].message.content

    if not raw_text or not raw_text.strip():
        print(
            "[QUERY REWRITE WARNING] "
            "DeepSeek returned empty content"
        )
        return query

    try:
        result = json.loads(raw_text)

        rewritten_query = result.get(
            "rewritten_query"
        )

        if not rewritten_query:
            return query

        return rewritten_query.strip()

    except Exception as exc:
        print(
            f"[QUERY REWRITE WARNING] {exc}"
        )

        # Never break retrieval just because rewriting failed.
        return query

In [45]:
conversation_history = [
    # {
    #     "role": "user",
    #     "content": "What can I see in Hoi An?"
    # },
    # {
    #     "role": "assistant",
    #     "content": "Hoi An has cultural sites, food and..."
    # },
]

# query = "and something near there for families?"
query = "best cultural attractions in Hue"

rewritten_query = rewrite_query(
    query=query,
    conversation_history=conversation_history,
)

print(rewritten_query)

best cultural attractions in Hue


## 4. Semantic Query Parsing

In [46]:
def get_known_cities() -> list[str]:
    db = SessionLocal()

    try:
        rows = (
            db.query(RagChunkORM.city)
            .filter(RagChunkORM.city.isnot(None))
            .filter(RagChunkORM.city != "")
            .distinct()
            .order_by(RagChunkORM.city)
            .all()
        )

        return [
            row[0]
            for row in rows
            if row[0]
        ]

    finally:
        db.close()

known_cities = get_known_cities()

In [47]:
from config.vocab import (
    ALLOWED_ACTIVITIES,
    ALLOWED_PLACE_TYPES,
    ALLOWED_SUITABLE_FOR,
    ALLOWED_TRAVEL_STYLES,
)

In [48]:
def build_query_parser_prompt(
    known_cities: list[str],
) -> str:
    return f"""
Extract travel retrieval metadata from the user query.

Return JSON only.

Cities: {", ".join(known_cities)}
Place types: {", ".join(ALLOWED_PLACE_TYPES)}
Activities: {", ".join(ALLOWED_ACTIVITIES)}
Travel styles: {", ".join(ALLOWED_TRAVEL_STYLES)}
Suitable for: {", ".join(ALLOWED_SUITABLE_FOR)}

Intent:
itinerary, recommendation, attraction_search,
accommodation_search, food_search, transport,
event_search, travel_information

Rules:
- Use only values from the lists.
- City must exactly match a listed city.
- Only extract information stated or strongly implied by the query.
- Do not infer preferences from the destination itself.
- Unknown scalar = null.
- Unknown list = [].

JSON:
{{
  "intent": null,
  "location": {{
    "country": null,
    "city": null,
    "province": null
  }},
  "place_types": [],
  "activities": [],
  "travel_styles": [],
  "suitable_for": [],
  "constraints": {{
    "budget": null,
    "duration_days": null,
    "date_from": null,
    "date_to": null,
    "near_place": null,
    "max_distance_km": null
  }}
}}
""".strip()

In [49]:
import json
import time

def parse_query_deepseek(
    query: str,
    model: str = DEEPSEEK_METADATA_MODEL,
) -> ParsedQuery:
    """
    Parse a natural-language travel query into structured retrieval metadata.
    """

    start = time.perf_counter()

    # =====================================================
    # INPUT VALIDATION
    # =====================================================

    if not query or not query.strip():
        raise ValueError("Query cannot be empty")

    # =====================================================
    # GET KNOWN CITIES
    # =====================================================

    known_cities = get_known_cities()

    db_time = time.perf_counter()

    # =====================================================
    # BUILD PROMPT
    # =====================================================

    system_msg = build_query_parser_prompt(
        known_cities
    )

    prompt_time = time.perf_counter()

    # =====================================================
    # DEEPSEEK CALL
    # =====================================================

    deepseek_client = get_deepseek_client()

    api_start = time.perf_counter()

    response = deepseek_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_msg,
            },
            {
                "role": "user",
                "content": query.strip(),
            },
        ],
        temperature=0,
        max_tokens=300,
        response_format={
            "type": "json_object",
        },
    )

    api_end = time.perf_counter()

    # =====================================================
    # GET RESPONSE TEXT
    # =====================================================

    raw_text = response.choices[0].message.content

    if not raw_text or not raw_text.strip():
        raise ValueError(
            "DeepSeek returned empty query parsing result"
        )

    # =====================================================
    # JSON PARSING
    # =====================================================

    try:
        parsed_json = json.loads(raw_text)

    except json.JSONDecodeError as exc:
        raise ValueError(
            f"DeepSeek returned invalid JSON: {exc}\n"
            f"Response: {raw_text[:1500]}"
        ) from exc

    # =====================================================
    # PYDANTIC VALIDATION
    # =====================================================

    parsed = ParsedQuery.model_validate(
        parsed_json
    )

    # =====================================================
    # FINAL SAFETY VALIDATION
    # =====================================================

    # City must exist in DB
    if (
        parsed.location.city
        and parsed.location.city not in known_cities
    ):
        print(
            "[QUERY PARSER WARNING] "
            f"Invalid city returned: {parsed.location.city}"
        )

        parsed.location.city = None

    # Remove unsupported place types
    parsed.place_types = [
        value
        for value in parsed.place_types
        if value in ALLOWED_PLACE_TYPES
    ]

    # Remove unsupported activities
    parsed.activities = [
        value
        for value in parsed.activities
        if value in ALLOWED_ACTIVITIES
    ]

    # Remove unsupported travel styles
    parsed.travel_styles = [
        value
        for value in parsed.travel_styles
        if value in ALLOWED_TRAVEL_STYLES
    ]

    # Remove unsupported suitable_for values
    parsed.suitable_for = [
        value
        for value in parsed.suitable_for
        if value in ALLOWED_SUITABLE_FOR
    ]

    # =====================================================
    # TIMING
    # =====================================================

    end = time.perf_counter()

    print("\n[QUERY PARSER TIMING]")
    print(f"DB:              {db_time - start:.3f}s")
    print(f"Prompt build:    {prompt_time - db_time:.3f}s")
    print(f"DeepSeek API:    {api_end - api_start:.3f}s")
    print(f"Post-processing: {end - api_end:.3f}s")
    print(f"TOTAL:           {end - start:.3f}s")

    return parsed

In [50]:
def parse_query(
    query: str,
) -> ParsedQuery:
    try:
        return parse_query_deepseek(query)

    except Exception as exc:
        print(f"[QUERY PARSER WARNING] {exc}")

        return ParsedQuery(
            intent="travel_information"
        )

In [51]:
parsed_query = parse_query(
    rewritten_query
)

parsed_query.model_dump()


[QUERY PARSER TIMING]
DB:              0.008s
Prompt build:    0.001s
DeepSeek API:    1.187s
Post-processing: 0.001s
TOTAL:           1.197s


{'intent': 'attraction_search',
 'location': {'country': None, 'city': 'Hue', 'province': None},
 'place_types': ['attraction'],
 'activities': [],
 'travel_styles': ['culture'],
 'suitable_for': [],
 'constraints': {'budget': None,
  'duration_days': None,
  'date_from': None,
  'date_to': None,
  'near_place': None,
  'max_distance_km': None}}

## 5. User Memory Retrieval

In [52]:
def get_user_memory(
    user_id: Optional[str],
) -> UserTravelMemory:

    if not user_id:
        return UserTravelMemory()

    # Replace later with actual UserMemoryORM retrieval.
    return UserTravelMemory(
        preferred_travel_styles=[
            "culture",
            "food",
        ],
        preferred_activities=[
            "sightseeing",
        ],
        budget_level="mid_range",
    )

In [53]:
user_memory = get_user_memory(
    user_id="test-user"
)

user_memory.model_dump()

{'preferred_travel_styles': ['culture', 'food'],
 'preferred_activities': ['sightseeing'],
 'budget_level': 'mid_range',
 'avoid': []}

## 6. Metadata-aware Filter Construction

In [54]:
@dataclass
class RetrievalFilters:
    city: Optional[str] = None
    province: Optional[str] = None
    country: Optional[str] = None

    place_types: list[str] = field(
        default_factory=list
    )


def build_retrieval_filters(
    parsed: ParsedQuery,
) -> RetrievalFilters:

    return RetrievalFilters(
        country=parsed.location.country,
        city=parsed.location.city,
        province=parsed.location.province,
        place_types=parsed.place_types,
    )

In [55]:
filters = build_retrieval_filters(
    parsed_query
)

filters

RetrievalFilters(city='Hue', province=None, country=None, place_types=['attraction'])

## 7. DB Row → LangChain Document

In [56]:
def row_to_document(row) -> Document:

    return Document(
        page_content=row.chunk_text,

        metadata={
            "chunk_id": str(row.id),

            "document_id": (
                str(row.document_id)
                if getattr(row, "document_id", None)
                else None
            ),

            "country": getattr(row, "country", None),
            "city": getattr(row, "city", None),
            "province": getattr(row, "province", None),

            "place_name": getattr(row, "place_name", None),
            "place_type": getattr(row, "place_type", None),

            "chunk_topic": getattr(row, "ai_topic", None),

            "ai_summary": getattr(row, "ai_summary", None),

            "ai_tags": getattr(row, "ai_tags", None) or [],
            "ai_activities": getattr(row, "ai_activities", None) or [],
            "ai_travel_styles": getattr(row, "ai_travel_styles", None) or [],
            "ai_suitable_for": getattr(row, "ai_suitable_for", None) or [],

            "distance": (
                float(row.distance)
                if hasattr(row, "distance")
                and row.distance is not None
                else None
            ),

            "source_date": getattr(row, "source_date", None),
            "updated_at": getattr(row, "updated_at", None),
            "latitude": getattr(row, "latitude", None),
            "longitude": getattr(row, "longitude", None),
        },
    )

## 8. Vector Search

In [57]:
def vector_search(
    query: str,
    filters: RetrievalFilters,
    limit: int = 30,
) -> list[Document]:

    db: Session = SessionLocal()

    try:
        query_embedding = embedding_model.encode(query).tolist()

        where_parts = ["embedding IS NOT NULL"]

        params = {
            "embedding": query_embedding,
            "limit": limit,
        }

        if filters.city:
            where_parts.append(
                "LOWER(city) = LOWER(:city)"
            )
            params["city"] = filters.city

        if filters.province:
            where_parts.append(
                "LOWER(province) = LOWER(:province)"
            )
            params["province"] = filters.province

        where_sql = " AND ".join(
            where_parts
        )

        sql = text(
            f"""
            SELECT
                *,
                embedding <=> CAST(:embedding AS vector)
                    AS distance
            FROM rag_chunks
            WHERE {where_sql}
            ORDER BY
                embedding <=> CAST(:embedding AS vector)
            LIMIT :limit
            """
        )

        rows = db.execute(sql, params).fetchall()

        return [ row_to_document(row) for row in rows]

    finally:
        db.close()

In [58]:
vector_results = vector_search(
    query=rewritten_query,
    filters=filters,
    limit=30,
)

print("Vector results:", len(vector_results))

for i, doc in enumerate(vector_results[:5], start=1):
    print("=" * 70)
    print("Rank:", i)
    print(doc.metadata)
    print(doc.page_content[:300])

Vector results: 30
Rank: 1
{'chunk_id': 'aa50263b-b9ba-49b3-b40b-26683743c3d6', 'document_id': '4ef00143-ced5-4f58-980d-c7f00faa94a6', 'country': 'Vietnam', 'city': 'Hue', 'province': None, 'place_name': None, 'place_type': None, 'chunk_topic': 'accommodation', 'ai_summary': 'This chunk introduces the accommodation options in Hue, noting a mix of cheap, mid-market, and expensive hotels, with a cluster around a short lane.', 'ai_tags': ['budget', 'hotels'], 'ai_activities': ['sleeping'], 'ai_travel_styles': ['budget'], 'ai_suitable_for': ['budget_travelers'], 'distance': 0.391388929403487, 'source_date': None, 'updated_at': datetime.datetime(2026, 8, 6, 2, 41, 39, 829377, tzinfo=datetime.timezone(datetime.timedelta(seconds=25200))), 'latitude': None, 'longitude': None}
Section: Sleep

[[edit](https://en.wikivoyage.org/w/index.php?title=Hue&action=edit§ion=30)]  
There are plenty of cheap hotels and mid-market hotels in Hue, as well as a couple of expensive giants. The largest cluster is

## 9. BM25 Search

In [59]:
def bm25_search(
    query: str,
    filters: RetrievalFilters,
    limit: int = 30,
) -> list[Document]:

    db: Session = SessionLocal()

    try:
        query_builder = db.query(RagChunkORM)

        # ==========================
        # HARD FILTERS
        # ==========================

        if filters.country:
            query_builder = query_builder.filter(
                RagChunkORM.country.ilike(filters.country)
            )

        if filters.city:
            query_builder = query_builder.filter(
                RagChunkORM.city.ilike(filters.city)
            )

        if filters.province:
            query_builder = query_builder.filter(
                RagChunkORM.province.ilike(filters.province)
            )

        if filters.place_types:
            query_builder = query_builder.filter(
                RagChunkORM.place_type.in_(filters.place_types)
            )

        rows = query_builder.all()

        if not rows:
            return []

        # ==========================
        # BUILD BM25 DOCUMENTS
        # ==========================

        documents = []

        for row in rows:
            original_doc = row_to_document(row)

            # Text specifically used for BM25
            bm25_text_parts = []

            if getattr(row, "place_name", None):
                bm25_text_parts.append(
                    f"Place: {row.place_name}"
                )

            if getattr(row, "ai_summary", None):
                bm25_text_parts.append(
                    row.ai_summary
                )

            if getattr(row, "chunk_text", None):
                bm25_text_parts.append(
                    row.chunk_text
                )

            bm25_text = "\n".join(
                bm25_text_parts
            )

            documents.append(
                Document(
                    page_content=bm25_text,
                    metadata=original_doc.metadata,
                )
            )

        # ==========================
        # BM25
        # ==========================

        retriever = BM25Retriever.from_documents(
            documents
        )

        retriever.k = limit

        return retriever.invoke(query)

    finally:
        db.close()

In [60]:
bm25_results = bm25_search(
    query=rewritten_query,
    filters=filters,
    limit=30,
)

print("BM25 results:", len(bm25_results))

for i, doc in enumerate(bm25_results[:5], start=1):
    print("=" * 70)
    print("Rank:", i)
    print(doc.metadata)
    print(doc.page_content[:300])

BM25 results: 12
Rank: 1
{'chunk_id': 'ffa6560a-68be-4548-9497-4f4da10b6bdb', 'document_id': '4ef00143-ced5-4f58-980d-c7f00faa94a6', 'country': 'Vietnam', 'city': 'Hue', 'province': None, 'place_name': 'Tombs of the Emperors', 'place_type': 'attraction', 'chunk_topic': 'overview', 'ai_summary': 'Overview of the Tombs of the Emperors in Hue, including location, access, and historical context.', 'ai_tags': ['boat_tour', 'buddhist_architecture', 'imperial_tombs', 'perfume_river'], 'ai_activities': ['cruising', 'sightseeing', 'exploring_tombs'], 'ai_travel_styles': ['culture', 'history', 'relaxation'], 'ai_suitable_for': ['culture_seekers', 'first_time_visitors', 'history_lovers'], 'distance': None, 'source_date': None, 'updated_at': datetime.datetime(2026, 8, 10, 21, 39, 53, 599436, tzinfo=datetime.timezone(datetime.timedelta(seconds=25200))), 'latitude': None, 'longitude': None}
Place: Tombs of the Emperors
Overview of the Tombs of the Emperors in Hue, including location, access, and his

## 10. Metadata Soft Boost

In [61]:
def metadata_boost(
    doc: Document,
    parsed: ParsedQuery,
    memory: UserTravelMemory,
) -> float:

    score = 0.0

    metadata = doc.metadata

    doc_styles = set(
        metadata.get(
            "ai_travel_styles",
            [],
        )
    )

    doc_activities = set(
        metadata.get(
            "ai_activities",
            [],
        )
    )

    doc_suitable = set(
        metadata.get(
            "ai_suitable_for",
            [],
        )
    )

    # Explicit query preference
    for style in parsed.travel_styles:
        if style in doc_styles:
            score += 0.015

    for activity in parsed.activities:
        if activity in doc_activities:
            score += 0.015

    for suitable in parsed.suitable_for:
        if suitable in doc_suitable:
            score += 0.015

    # User memory = weaker boost
    for style in memory.preferred_travel_styles:
        if style in doc_styles:
            score += 0.005

    return score

## 11. Geo Boost

In [62]:
def haversine_distance_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:

    earth_radius = 6371.0

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    delta_lat = lat2 - lat1
    delta_lon = math.radians(
        lon2 - lon1
    )

    a = (
        math.sin(delta_lat / 2) ** 2
        +
        math.cos(lat1)
        * math.cos(lat2)
        * math.sin(delta_lon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a),
    )

    return earth_radius * c


def geo_boost(
    doc: Document,
    target_lat: Optional[float],
    target_lon: Optional[float],
) -> float:

    if target_lat is None or target_lon is None:
        return 0.0

    lat = doc.metadata.get("latitude")
    lon = doc.metadata.get("longitude")

    if lat is None or lon is None:
        return 0.0

    distance = haversine_distance_km(
        target_lat,
        target_lon,
        float(lat),
        float(lon),
    )

    doc.metadata["geo_distance_km"] = distance

    return 0.02 / (1 + distance)

## 12. Freshness Boost

In [63]:
def freshness_boost(
    doc: Document,
    max_boost: float = 0.01,
) -> float:

    date_value = (
        doc.metadata.get("source_date")
        or doc.metadata.get("updated_at")
    )

    if not date_value:
        return 0.0

    if isinstance(date_value, str):
        try:
            date_value = datetime.fromisoformat(
                date_value.replace(
                    "Z",
                    "+00:00",
                )
            )
        except ValueError:
            return 0.0

    if date_value.tzinfo is None:
        date_value = date_value.replace(
            tzinfo=timezone.utc
        )

    now = datetime.now(
        timezone.utc
    )

    age_days = max(
        (now - date_value).days,
        0,
    )

    decay = math.exp(
        -age_days / 365
    )

    return max_boost * decay

## 13. Score Fusion (RRF + Boosts)

In [64]:
def fuse_results(
    vector_docs: list[Document],
    bm25_docs: list[Document],
    parsed: ParsedQuery,
    memory: UserTravelMemory,
    rrf_k: int = 60,
    target_lat: Optional[float] = None,
    target_lon: Optional[float] = None,
) -> list[Document]:

    scored: dict[str, dict] = {}

    # --------------------------
    # VECTOR
    # --------------------------

    for rank, doc in enumerate(
        vector_docs,
        start=1,
    ):
        chunk_id = doc.metadata[
            "chunk_id"
        ]

        if chunk_id not in scored:
            scored[chunk_id] = {
                "doc": doc,
                "score": 0.0,
                "vector_rank": None,
                "bm25_rank": None,
            }

        scored[chunk_id][
            "score"
        ] += 1 / (rrf_k + rank)

        scored[chunk_id][
            "vector_rank"
        ] = rank

    # --------------------------
    # BM25
    # --------------------------

    for rank, doc in enumerate(
        bm25_docs,
        start=1,
    ):
        chunk_id = doc.metadata[
            "chunk_id"
        ]

        if chunk_id not in scored:
            scored[chunk_id] = {
                "doc": doc,
                "score": 0.0,
                "vector_rank": None,
                "bm25_rank": None,
            }

        scored[chunk_id][
            "score"
        ] += 1 / (rrf_k + rank)

        scored[chunk_id][
            "bm25_rank"
        ] = rank

    # --------------------------
    # BOOSTS
    # --------------------------

    for item in scored.values():
        doc = item["doc"]

        meta_score = metadata_boost(
            doc=doc,
            parsed=parsed,
            memory=memory,
        )

        geo_score = geo_boost(
            doc=doc,
            target_lat=target_lat,
            target_lon=target_lon,
        )

        fresh_score = freshness_boost(
            doc=doc,
        )

        item["score"] += (
            meta_score
            + geo_score
            + fresh_score
        )

        item["metadata_boost"] = meta_score
        item["geo_boost"] = geo_score
        item["freshness_boost"] = fresh_score

    ranked = sorted(
        scored.values(),
        key=lambda item: item["score"],
        reverse=True,
    )

    results = []

    for item in ranked:
        doc = item["doc"]

        doc.metadata["fusion_score"] = item["score"]
        doc.metadata["vector_rank"] = item["vector_rank"]
        doc.metadata["bm25_rank"] = item["bm25_rank"]
        doc.metadata["metadata_boost"] = item["metadata_boost"]
        doc.metadata["geo_boost"] = item["geo_boost"]
        doc.metadata["freshness_boost"] = item["freshness_boost"]

        results.append(doc)

    return results

In [65]:
fused_results = fuse_results(
    vector_docs=vector_results,
    bm25_docs=bm25_results,
    parsed=parsed_query,
    memory=user_memory,
)

print("Fused results:", len(fused_results))

for i, doc in enumerate(fused_results[:10], start=1):
    print("=" * 70)
    print("Fusion rank:", i)
    print({
        "chunk_id": doc.metadata.get("chunk_id"),
        "vector_rank": doc.metadata.get("vector_rank"),
        "bm25_rank": doc.metadata.get("bm25_rank"),
        "fusion_score": doc.metadata.get("fusion_score"),
        "metadata_boost": doc.metadata.get("metadata_boost"),
        "geo_boost": doc.metadata.get("geo_boost"),
        "freshness_boost": doc.metadata.get("freshness_boost"),
    })

Fused results: 37
Fusion rank: 1
{'chunk_id': '52d9882a-7864-4781-977f-c9dff787e904', 'vector_rank': 6, 'bm25_rank': 6, 'fusion_score': 0.0603030303030303, 'metadata_boost': 0.02, 'geo_boost': 0.0, 'freshness_boost': 0.01}
Fusion rank: 2
{'chunk_id': '73d2425e-7d88-40a1-acbd-59294471caa3', 'vector_rank': 9, 'bm25_rank': 4, 'fusion_score': 0.06011775362318841, 'metadata_boost': 0.02, 'geo_boost': 0.0, 'freshness_boost': 0.01}
Fusion rank: 3
{'chunk_id': 'ffa6560a-68be-4548-9497-4f4da10b6bdb', 'vector_rank': 14, 'bm25_rank': 1, 'fusion_score': 0.05990695613646434, 'metadata_boost': 0.02, 'geo_boost': 0.0, 'freshness_boost': 0.01}
Fusion rank: 4
{'chunk_id': '547d0a64-8b17-42ce-9d10-282a84097ac1', 'vector_rank': 11, 'bm25_rank': 11, 'fusion_score': 0.058169014084507045, 'metadata_boost': 0.02, 'geo_boost': 0.0, 'freshness_boost': 0.01}
Fusion rank: 5
{'chunk_id': '9c159f84-e6f3-43d2-9516-8a4d2096a7b0', 'vector_rank': 2, 'bm25_rank': None, 'fusion_score': 0.05112903225806452, 'metadata_boo

## 14. Top-K Pruning

In [66]:
fusion_top_k = 20

candidates = fused_results[:fusion_top_k]

print(
    "Candidates before reranking:",
    len(candidates),
)

Candidates before reranking: 20


## 15. CrossEncoder Reranking

In [67]:
def rerank_documents(
    query: str,
    documents: list[Document],
    top_k: int = 8,
) -> list[Document]:

    if not documents:
        return []

    pairs = [
        (
            query,
            doc.page_content,
        )
        for doc in documents
    ]

    scores = reranker.predict(
        pairs
    )

    scored = []

    for doc, score in zip(
        documents,
        scores,
    ):
        score = float(score)

        doc.metadata[
            "rerank_score"
        ] = score

        scored.append(
            (doc, score)
        )

    scored.sort(
        key=lambda item: item[1],
        reverse=True,
    )

    return [
        doc
        for doc, _ in scored[:top_k]
    ]

In [68]:
reranked_results = rerank_documents(
    query=rewritten_query,
    documents=candidates,
    top_k=8,
)

for rank, doc in enumerate(reranked_results, start=1):
    print("=" * 70)

    print(f"Rank: {rank}")
    print("Place:", doc.metadata.get("place_name"))
    print("Vector rank:", doc.metadata.get("vector_rank"))
    print("BM25 rank:", doc.metadata.get("bm25_rank"))
    print("Fusion:", doc.metadata.get("fusion_score"))
    print("Rerank:", doc.metadata.get("rerank_score"))

    print(doc.page_content[:400])

Rank: 1
Place: Tombs of the Emperors
Vector rank: 14
BM25 rank: 1
Fusion: 0.05990695613646434
Rerank: 5.496091842651367
Section: See > Tombs of the Emperors

[[edit](https://en.wikivoyage.org/w/index.php?title=Hue&action=edit§ion=16)]  
Another of Hue's great attractions are the Tombs of the Emperors, on the Perfume River south of the city. They are accessible by taxi or bike from the city, but the best way to see them is to hire a river boat and go for a cruise. Plan to make a full day of it. Most of the tombs are
Rank: 2
Place: Imperial City of Hue
Vector rank: 9
BM25 rank: 4
Fusion: 0.06011775362318841
Rerank: 3.7633771896362305
Section: See > Imperial Citadel

[4](https://en.wikivoyage.org/wiki/Special:Map/17/16.46955/107.57782/en)Imperial City of Hue (Đại Nội). Daily, 06:30-17:00. The former imperial seat of government and Hue's prime attraction, this is a great sprawling complex of temples, pavilions, moats, walls, gates, shops, museums and galleries, featuring art and costumes f

## 16. Retrieval Confidence

In [69]:
class RetrievalConfidence(BaseModel):
    level: str
    score: float
    evidence_count: int
    top_score: Optional[float] = None
    score_gap: Optional[float] = None

In [70]:
def evaluate_retrieval_confidence(
    documents: list[Document],
) -> RetrievalConfidence:

    if not documents:
        return RetrievalConfidence(
            level="low",
            score=0.0,
            evidence_count=0,
        )

    rerank_scores = [
        doc.metadata.get(
            "rerank_score"
        )
        for doc in documents
    ]

    rerank_scores = [
        float(score)
        for score in rerank_scores
        if score is not None
    ]

    if not rerank_scores:
        return RetrievalConfidence(
            level="low",
            score=0.2,
            evidence_count=len(documents),
        )

    top_score = rerank_scores[0]

    if len(rerank_scores) > 1:
        score_gap = (
            rerank_scores[0]
            - rerank_scores[1]
        )
    else:
        score_gap = 0.0

    array = np.array(
        rerank_scores,
        dtype=float,
    )

    exp_scores = np.exp(
        array - np.max(array)
    )

    probabilities = (
        exp_scores
        / exp_scores.sum()
    )

    top_share = float(
        probabilities[0]
    )

    evidence_factor = min(
        len(documents) / 5,
        1.0,
    )

    confidence = (
        0.7 * top_share
        +
        0.3 * evidence_factor
    )

    if confidence >= 0.7:
        level = "high"
    elif confidence >= 0.4:
        level = "medium"
    else:
        level = "low"

    return RetrievalConfidence(
        level=level,
        score=round(
            confidence,
            4,
        ),
        evidence_count=len(documents),
        top_score=top_score,
        score_gap=score_gap,
    )

In [71]:
retrieval_confidence = (
    evaluate_retrieval_confidence(
        reranked_results
    )
)

retrieval_confidence.model_dump()

{'level': 'high',
 'score': 0.8776,
 'evidence_count': 8,
 'top_score': 5.496091842651367,
 'score_gap': 1.7327146530151367}

## 17. Evidence Construction

In [72]:
class EvidenceItem(BaseModel):
    evidence_id: str
    chunk_id: str
    document_id: Optional[str]
    place_name: Optional[str]
    content: str
    metadata: Dict[str, Any]


def build_evidence(
    documents: list[Document],
) -> list[EvidenceItem]:

    evidence = []

    for index, doc in enumerate(
        documents,
        start=1,
    ):
        evidence.append(
            EvidenceItem(
                evidence_id=f"E{index}",
                chunk_id=doc.metadata["chunk_id"],
                document_id=doc.metadata.get(
                    "document_id"
                ),
                place_name=doc.metadata.get(
                    "place_name"
                ),
                content=doc.page_content,
                metadata=doc.metadata,
            )
        )

    return evidence

In [73]:
evidence = build_evidence(
    reranked_results
)

[e.model_dump() for e in evidence]

[{'evidence_id': 'E1',
  'chunk_id': 'ffa6560a-68be-4548-9497-4f4da10b6bdb',
  'document_id': '4ef00143-ced5-4f58-980d-c7f00faa94a6',
  'place_name': 'Tombs of the Emperors',
  'content': "Section: See > Tombs of the Emperors\n\n[[edit](https://en.wikivoyage.org/w/index.php?title=Hue&action=edit§ion=16)]  \nAnother of Hue's great attractions are the Tombs of the Emperors, on the Perfume River south of the city. They are accessible by taxi or bike from the city, but the best way to see them is to hire a river boat and go for a cruise. Plan to make a full day of it. Most of the tombs are open from 07:30 or 08:00 to 17:30, depending on the season.  \nThe tombs themselves are worth the cost and effort. They mostly date from the late 19th or early 20th centuries, when the emperors had been reduced to figureheads under French colonial rule and had little else to do than build themselves elaborate tombs. The finest of them are the Tomb of Tu Duc, the Tomb of Minh Mang and the Tomb of Khai Din

## 18. Conditional Tool Planning

In [74]:
class ToolPlan(BaseModel):
    need_weather: bool = False
    need_maps: bool = False
    need_events: bool = False


def create_tool_plan(parsed: ParsedQuery) -> ToolPlan:

    plan = ToolPlan()

    if (
        parsed.constraints.date_from
        or parsed.constraints.date_to
    ):
        plan.need_weather = True

    if parsed.intent == "itinerary":
        plan.need_maps = True

    if parsed.intent == "events":
        plan.need_events = True

    return plan

In [75]:
tool_plan = create_tool_plan(
    parsed_query
)

tool_plan.model_dump()

{'need_weather': False, 'need_maps': False, 'need_events': False}

## 19. Structured Itinerary Drafting

This is currently a mock planner so you can test the full flow.
Replace this later with your DeepSeek structured JSON planner.

In [76]:
class ItineraryStop(BaseModel):
    place_name: str
    time_slot: Optional[str] = None
    activity: Optional[str] = None
    reason: Optional[str] = None

    evidence_ids: list[str] = Field(
        default_factory=list
    )


class ItineraryDay(BaseModel):
    day: int
    theme: Optional[str] = None

    stops: list[ItineraryStop] = Field(
        default_factory=list
    )


class StructuredItinerary(BaseModel):
    destination: Optional[str] = None

    days: list[ItineraryDay] = Field(
        default_factory=list
    )

In [77]:
def draft_itinerary_mock(
    parsed: ParsedQuery,
    evidence: list[EvidenceItem],
) -> StructuredItinerary:

    number_of_days = (
        parsed.constraints.duration_days
        or 1
    )

    destination = (parsed.location.city)

    days = []
    evidence_index = 0

    for day_number in range(
        1,
        number_of_days + 1,
    ):
        stops = []

        for _ in range(2):
            if evidence_index >= len(
                evidence
            ):
                break

            item = evidence[
                evidence_index
            ]

            evidence_index += 1
            stops.append(
                ItineraryStop(
                    place_name=(
                        item.place_name
                        or "Recommended place"
                    ),
                    evidence_ids=[
                        item.evidence_id
                    ],
                )
            )

        days.append(
            ItineraryDay(
                day=day_number,
                stops=stops,
            )
        )

    return StructuredItinerary(
        destination=destination,
        days=days,
    )

In [78]:
itinerary = draft_itinerary_mock(
    parsed=parsed_query,
    evidence=evidence,
)

itinerary.model_dump()

{'destination': 'Hue',
 'days': [{'day': 1,
   'theme': None,
   'stops': [{'place_name': 'Tombs of the Emperors',
     'time_slot': None,
     'activity': None,
     'reason': None,
     'evidence_ids': ['E1']},
    {'place_name': 'Imperial City of Hue',
     'time_slot': None,
     'activity': None,
     'reason': None,
     'evidence_ids': ['E2']}]}]}

## 20. Citation Extraction

In [79]:
class Citation(BaseModel):
    evidence_id: str
    chunk_id: str
    document_id: Optional[str]
    place_name: Optional[str]
    source_location: Optional[str] = None


def extract_citations(
    itinerary: StructuredItinerary,
    evidence: list[EvidenceItem],
) -> list[Citation]:

    evidence_map = {
        item.evidence_id: item
        for item in evidence
    }

    used_ids = set()

    for day in itinerary.days:
        for stop in day.stops:
            used_ids.update(
                stop.evidence_ids
            )

    citations = []

    for evidence_id in used_ids:
        item = evidence_map.get(
            evidence_id
        )

        if not item:
            continue

        citations.append(
            Citation(
                evidence_id=evidence_id,
                chunk_id=item.chunk_id,
                document_id=item.document_id,
                place_name=item.place_name,
                source_location=(
                    item.metadata.get(
                        "source_location"
                    )
                ),
            )
        )

    return citations

In [80]:
citations = extract_citations(
    itinerary=itinerary,
    evidence=evidence,
)

[c.model_dump() for c in citations]

[{'evidence_id': 'E1',
  'chunk_id': 'ffa6560a-68be-4548-9497-4f4da10b6bdb',
  'document_id': '4ef00143-ced5-4f58-980d-c7f00faa94a6',
  'place_name': 'Tombs of the Emperors',
  'source_location': None},
 {'evidence_id': 'E2',
  'chunk_id': '73d2425e-7d88-40a1-acbd-59294471caa3',
  'document_id': '4ef00143-ced5-4f58-980d-c7f00faa94a6',
  'place_name': 'Imperial City of Hue',
  'source_location': None}]

## 21. Final Answer Renderer

In [81]:
def render_itinerary(
    itinerary: StructuredItinerary,
) -> str:

    lines = []

    if itinerary.destination:
        lines.append(
            f"Trip to {itinerary.destination}"
        )

    for day in itinerary.days:
        lines.append(
            f"\nDay {day.day}"
        )

        for stop in day.stops:
            citation_text = ""

            if stop.evidence_ids:
                citation_text = (
                    " ["
                    + ", ".join(
                        stop.evidence_ids
                    )
                    + "]"
                )

            lines.append(
                f"- {stop.place_name}"
                f"{citation_text}"
            )

    return "\n".join(
        lines
    )

In [82]:
answer = render_itinerary(
    itinerary
)

print(answer)

Trip to Hue

Day 1
- Tombs of the Emperors [E1]
- Imperial City of Hue [E2]


## 22. Frontend Response Schema

In [83]:
class TravelAssistantResponse(BaseModel):
    original_query: str
    rewritten_query: str
    parsed_query: Dict[str, Any]

    itinerary: Optional[
        StructuredItinerary
    ]

    sources: list[Citation]

    retrieval_confidence: (
        RetrievalConfidence
    )

    answer: str

In [84]:
response = TravelAssistantResponse(
    original_query=query,
    rewritten_query=rewritten_query,
    parsed_query=parsed_query.model_dump(),
    itinerary=itinerary,
    sources=citations,
    retrieval_confidence=retrieval_confidence,
    answer=answer,
)

response.model_dump()

{'original_query': 'best cultural attractions in Hue',
 'rewritten_query': 'best cultural attractions in Hue',
 'parsed_query': {'intent': 'attraction_search',
  'location': {'country': None, 'city': 'Hue', 'province': None},
  'place_types': ['attraction'],
  'activities': [],
  'travel_styles': ['culture'],
  'suitable_for': [],
  'constraints': {'budget': None,
   'duration_days': None,
   'date_from': None,
   'date_to': None,
   'near_place': None,
   'max_distance_km': None}},
 'itinerary': {'destination': 'Hue',
  'days': [{'day': 1,
    'theme': None,
    'stops': [{'place_name': 'Tombs of the Emperors',
      'time_slot': None,
      'activity': None,
      'reason': None,
      'evidence_ids': ['E1']},
     {'place_name': 'Imperial City of Hue',
      'time_slot': None,
      'activity': None,
      'reason': None,
      'evidence_ids': ['E2']}]}]},
 'sources': [{'evidence_id': 'E1',
   'chunk_id': 'ffa6560a-68be-4548-9497-4f4da10b6bdb',
   'document_id': '4ef00143-ced5-4f58-

## 23. Full Pipeline Function

In [85]:
def run_travel_pipeline(
    query: str,
    conversation_history: list[dict] | None = None,
    user_id: Optional[str] = None,
):

    conversation_history = (
        conversation_history or []
    )

    # 1. Query rewrite
    rewritten_query = rewrite_query(
        query=query,
        conversation_history=conversation_history,
    )

    # 2. Query parsing
    parsed = parse_query(
        rewritten_query
    )

    # 3. User memory
    memory = get_user_memory(
        user_id
    )

    # 4. Metadata filter
    filters = build_retrieval_filters(
        parsed
    )

    # 5. Vector search
    vector_docs = vector_search(
        query=rewritten_query,
        filters=filters,
        limit=30,
    )

    # 6. BM25 search
    bm25_docs = bm25_search(
        query=rewritten_query,
        filters=filters,
        limit=30,
    )

    # 7. Fusion + boosts
    fused_docs = fuse_results(
        vector_docs=vector_docs,
        bm25_docs=bm25_docs,
        parsed=parsed,
        memory=memory,
    )

    # 8. Top-20 pruning
    candidates = fused_docs[:20]

    # 9. CrossEncoder reranking
    reranked_docs = rerank_documents(
        query=rewritten_query,
        documents=candidates,
        top_k=8,
    )

    # 10. Confidence
    confidence = (
        evaluate_retrieval_confidence(
            reranked_docs
        )
    )

    # 11. Evidence
    evidence = build_evidence(
        reranked_docs
    )

    # 12. Conditional tool plan
    tool_plan = create_tool_plan(
        parsed
    )

    # 13. Structured itinerary
    itinerary = draft_itinerary_mock(
        parsed=parsed,
        evidence=evidence,
    )

    # 14. Citations
    citations = extract_citations(
        itinerary=itinerary,
        evidence=evidence,
    )

    # 15. Final answer
    answer = render_itinerary(
        itinerary
    )

    return TravelAssistantResponse(
        original_query=query,
        rewritten_query=rewritten_query,
        parsed_query=parsed.model_dump(),
        itinerary=itinerary,
        sources=citations,
        retrieval_confidence=confidence,
        answer=answer,
    )

## 24. Full Pipeline Test

In [86]:
result = run_travel_pipeline(
    query=(
        "Plan family-friendly activities "
        "in Da Nang for 3 days. "
        "I like cultural attractions and food."
    ),
    user_id="test-user",
)

result.model_dump()


[QUERY PARSER TIMING]
DB:              0.010s
Prompt build:    0.000s
DeepSeek API:    1.445s
Post-processing: 0.000s
TOTAL:           1.454s


{'original_query': 'Plan family-friendly activities in Da Nang for 3 days. I like cultural attractions and food.',
 'rewritten_query': 'Plan family-friendly activities in Da Nang for 3 days. I like cultural attractions and food.',
 'parsed_query': {'intent': 'itinerary',
  'location': {'country': None, 'city': 'Da Nang', 'province': None},
  'place_types': ['attraction'],
  'activities': ['cultural_experience', 'food_tasting'],
  'travel_styles': ['culture', 'food'],
  'suitable_for': ['families', 'families_with_children'],
  'constraints': {'budget': None,
   'duration_days': 3,
   'date_from': None,
   'date_to': None,
   'near_place': None,
   'max_distance_km': None}},
 'itinerary': {'destination': 'Da Nang',
  'days': [{'day': 1,
    'theme': None,
    'stops': [{'place_name': 'Da Nang',
      'time_slot': None,
      'activity': None,
      'reason': None,
      'evidence_ids': ['E1']},
     {'place_name': 'Danang Marriott Resort & Spa, Non Nuoc Beach Villas',
      'time_slot': 

## 25. Recommended Development Order

For now, focus on validating:

**Query Parsing → Metadata Filter → Vector → BM25 → RRF Fusion → Top 20 → CrossEncoder → Confidence**

Only after the retrieval quality is good should you spend more time on:

- User memory
- Weather
- Maps
- Events
- DeepSeek itinerary planning
- Image search
- Final frontend rendering